══════════════════════════════════════════════════════════════
 WEEK 10  |  DAY 3  |  LANGCHAIN BASICS
══════════════════════════════════════════════════════════════

 LEARNING OBJECTIVES
 ───────────────────
 1. Understand what LangChain is and why it exists
 2. Create a simple chain: prompt → LLM → output
 3. Use prompt templates to build reusable prompts
 4. Chain multiple steps together using LCEL (LangChain Expression Language)
 5. Define tools and let the LLM call them (function calling)
 6. Maintain conversation history so the LLM remembers context (memory)
 7. Log and trace every LLM call for observability

 TIME:  ~65 minutes

 YOUTUBE
 ───────
 Search: "LangChain tutorial Python beginners 2025"
 Search: "LangChain LCEL chains explained"
 Search: "LangChain function calling tools tutorial"
 Search: "LangChain conversation memory tutorial"
 Search: "LangSmith tracing tutorial 2025"

 INSTALL:
   pip install langchain langchain-openai langchain-anthropic langchain-core

 NOTE: You need an API key set as an environment variable (from Day 1).

══════════════════════════════════════════════════════════════

In [ ]:

import os




 ─────────────────────────────────────────────────────────────
 ARCHITECTURE DECISION
 ─────────────────────
 Choosing between: raw API calls vs LangChain vs LlamaIndex vs LangGraph.
 Rule of thumb: use raw API for simple single-turn calls. Use LangChain for chains and tool calling. Use LlamaIndex for document retrieval pipelines. Move to LangGraph when you need stateful multi-step agents with conditional routing.

══════════════════════════════════════════════════════════════
 CONCEPT 1 — WHAT IS LANGCHAIN AND WHY USE IT?
══════════════════════════════════════════════════════════════

LangChain is a framework that makes building LLM applications easier.
Instead of writing raw API calls every time, LangChain gives you:
  - PromptTemplate  — reusable prompts with variables  ({topic}, {language})
  - LLM wrappers    — same interface for OpenAI, Anthropic, Gemini, etc.
  - Chains          — connect steps together: prompt | llm | parser
  - Memory          — keep conversation history
  - Tools           — let the LLM call functions (search, calculator, etc.)

Raw API call — works but not reusable:
  response = client.chat.completions.create(model=..., messages=[...])

LangChain chain — reusable and composable:
  chain = prompt_template | llm | output_parser
  result = chain.invoke({"topic": "data engineering"})

When to use LangChain vs raw API:
  Raw API   — simple one-off calls, full control, no extra dependencies
  LangChain — complex multi-step apps, RAG, agents, memory, tool use


EXAMPLE ──────────────────────────────────────────────────────


In [ ]:
print("=" * 55)
print("CONCEPT 1: LangChain Architecture")
print("=" * 55)
print()
print("Chain = PromptTemplate | LLM | OutputParser")
print()
print("  PromptTemplate:  'Translate {text} to {language}'")
print("       |")
print("  LLM (GPT-4o / Claude):  processes the filled-in prompt")
print("       |")
print("  OutputParser:  extracts the text from the response object")



══════════════════════════════════════════════════════════════
 EXERCISE 1
══════════════════════════════════════════════════════════════
For each use case, describe the 3 components of the chain as comments:
  PromptTemplate: what variables does it need?
  LLM: which model would you use? (fast=haiku/gpt-4o-mini, smart=claude/gpt-4o)
  OutputParser: what format do you want? (plain text, JSON, list)

Use case A: Summarize customer reviews for a product manager
Use case B: Convert a SQL query into plain English for a non-technical stakeholder
Use case C: Generate a Python function docstring given the function code

Expected output:
    3 chain designs as comments, each with PromptTemplate, LLM, OutputParser


══════════════════════════════════════════════════════════════
 CONCEPT 2 — PROMPT TEMPLATES
══════════════════════════════════════════════════════════════

A PromptTemplate is a prompt with placeholders that get filled in at runtime.
It keeps your prompts clean and reusable.


EXAMPLE ──────────────────────────────────────────────────────


In [ ]:

print()
print("=" * 55)
print("CONCEPT 2: Prompt Templates")
print("=" * 55)



from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.environ.get("OPENAI_API_KEY"))

# Create a reusable prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Give concise, direct answers."),
    ("user",   "{question}")
])

# Build the chain
chain = prompt | llm | StrOutputParser()

# Use the same chain for different roles and questions
print(chain.invoke({"role": "data engineer",  "question": "What is an ETL pipeline?"}))
print()
print(chain.invoke({"role": "data analyst",   "question": "What is a pivot table?"}))
print()
print(chain.invoke({"role": "ML engineer",    "question": "What is overfitting?"}))


In [ ]:

print("Template code shown above — uncomment once you have an API key.")
print()
print("Notice: same chain, different inputs — different context-aware answers.")



══════════════════════════════════════════════════════════════
 EXERCISE 2
══════════════════════════════════════════════════════════════
Build a chain called "classifier_chain" with this template:

  system: "You are a text classifier. Respond with ONLY one word."
  user:   "Classify this text as {category_type}: {text}"

Test it with:
  {"category_type": "sentiment (positive/negative/neutral)",
   "text": "The delivery was fast but the product broke after one day."}

  {"category_type": "support ticket type (billing/technical/shipping/other)",
   "text": "I was charged twice for my subscription this month."}

(Uncomment and test once you have an API key)

Expected output:
    negative
    billing


══════════════════════════════════════════════════════════════
 CONCEPT 3 — CHAINING MULTIPLE STEPS (LCEL)
══════════════════════════════════════════════════════════════

LCEL (LangChain Expression Language) lets you chain steps with the | operator.
Output of one step automatically becomes input of the next.

Example: a two-step chain
  Step 1: Extract key facts from a document
  Step 2: Turn those facts into a structured report


EXAMPLE ──────────────────────────────────────────────────────


In [ ]:

print()
print("=" * 55)
print("CONCEPT 3: Multi-Step Chain (LCEL)")
print("=" * 55)



from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.environ.get("OPENAI_API_KEY"))
parser = StrOutputParser()

# Step 1: extract facts
step1_prompt = ChatPromptTemplate.from_template(
    "Extract 3 key facts from this text as bullet points:\n{raw_text}"
)

# Step 2: turn facts into a summary
step2_prompt = ChatPromptTemplate.from_template(
    "Write a 2-sentence executive summary based on these facts:\n{facts}"
)

# Build the two-step chain
chain = (
    step1_prompt
    | llm
    | parser
    | (lambda facts: {"facts": facts})   # pass output to next step's variable
    | step2_prompt
    | llm
    | parser
)

result = chain.invoke({"raw_text": """
    Q3 Results: Revenue reached 2.4M, up 12% vs Q2.
    Top product: Laptop Pro generated 35% of total revenue.
    Challenge: 8% return rate due to logistics issues.
    Plan: hire 2 logistics managers, launch Laptop Pro 2 in Q4.
"""})
print("Final summary:")
print(result)


In [ ]:

print("Multi-step chain code shown above — uncomment once you have an API key.")



══════════════════════════════════════════════════════════════
 EXERCISE 3
══════════════════════════════════════════════════════════════
Design a 3-step LangChain pipeline for this data engineering task:

Input: a plain English description of a data transformation
  "I need a query that shows total revenue per city, only for cities
   with more than 10 orders, sorted from highest to lowest revenue."

Step 1: Convert the description to a SQL query
Step 2: Add comments to the SQL query explaining each part
Step 3: Rate the query for efficiency (1-5) with a brief reason

Write the 3 ChatPromptTemplate templates as strings.
Build the chain with LCEL (use the pattern shown above).
(Uncomment and test once you have a key)

Expected output:
    Step 1: a SQL SELECT with GROUP BY, HAVING, ORDER BY
    Step 2: the same SQL with inline comments
    Step 3: efficiency rating 1-5 with reasoning


In [ ]:

task_description = ("I need a query that shows total revenue per city, only for cities "
                    "with more than 10 orders, sorted from highest to lowest revenue.")






══════════════════════════════════════════════════════════════
 CONCEPT 4 — TOOL / FUNCTION CALLING
══════════════════════════════════════════════════════════════

Tool calling lets the LLM request to run a Python function you have defined.
This is how AI agents do real work beyond generating text.

HOW IT WORKS:
  1. You define tools — Python functions with clear docstrings and type hints
  2. You bind those tools to the LLM
  3. The LLM sees each tool's name, description, and parameter schema
  4. When you ask a question, the LLM returns a "call request": function name + arguments
  5. Your code runs the actual function with those arguments
  6. You can send the result back to the LLM for a final answer

IMPORTANT: the LLM does NOT run the function. It only returns a structured request.
Your code does the actual execution.

WHY THIS MATTERS:
  Without tools: LLM can only generate text based on training data
  With tools:    LLM can query a live database, call an API, run calculations

  Example data engineering tools:
    run_sql_query(query: str)          → execute SQL, return results
    fetch_pipeline_status(job_id: str) → check if an ETL job succeeded
    get_table_schema(table_name: str)  → return column names and types

INSTALL:
  pip install langchain-core langchain-openai

EXAMPLE ──────────────────────────────────────────────────────

In [ ]:

print()
print("=" * 55)
print("CONCEPT 4: Tool / Function Calling")
print("=" * 55)

# from langchain_core.tools import tool
# from langchain_openai import ChatOpenAI
#
# @tool
# def get_product_stock(product_name: str) -> str:
#     """Return the current stock level of a product by name."""
#     stock = {"laptop": 12, "monitor": 5, "keyboard": 34, "mouse": 67}
#     count = stock.get(product_name.lower(), 0)
#     return f"{product_name}: {count} units in stock"
#
# @tool
# def get_city_orders(city: str) -> str:
#     """Return the number of orders placed from a city this month."""
#     orders = {"tel aviv": 245, "haifa": 132, "jerusalem": 98}
#     count = orders.get(city.lower(), 0)
#     return f"{city}: {count} orders this month"
#
# llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.environ.get("OPENAI_API_KEY"))
# llm_with_tools = llm.bind_tools([get_product_stock, get_city_orders])
#
# question = "How many orders came from Haifa this month?"
# response = llm_with_tools.invoke(question)
#
# if response.tool_calls:
#     call = response.tool_calls[0]
#     print(f"LLM chose tool:  {call['name']}")
#     print(f"With arguments:  {call['args']}")
#     if call['name'] == 'get_city_orders':
#         result = get_city_orders.invoke(call['args'])
#         print(f"Tool result:     {result}")

print("Uncomment the tool calling code above once you have an API key.")
print()
print("How it works:")
print("  1. @tool reads the function name, type hints, and docstring")
print("  2. LLM sees the tool description and decides when to call it")
print("  3. LLM returns a call request — your code runs the actual function")
print("  4. One question can trigger multiple tool calls (search + calculate)")


══════════════════════════════════════════════════════════════
 EXERCISE 4
══════════════════════════════════════════════════════════════
Define 3 mock tools for a sales analytics assistant using the @tool decorator:
  get_top_product(category: str)      — returns the top-selling product in that category
  get_monthly_revenue(month: str)     — returns total revenue for that month
  compare_quarters(q1: str, q2: str)  — returns revenue comparison between two quarters

Use hardcoded dictionary values — no real database needed.
Each function must have a clear docstring (the LLM reads it to decide when to call the tool).

For each question in the questions list below:
  1. Write as a comment which tool would be called and with what arguments
  2. If you have an API key, bind the tools to an LLM and run all 3 questions to verify

Expected output:
    "What was the total revenue in March?"
      → get_monthly_revenue(month="March")  e.g. "March: 2,400,000 NIS"
    "What is the best-selling product in electronics?"
      → get_top_product(category="electronics")  e.g. "electronics: Laptop Pro"
    "How did Q1 compare to Q2 in revenue?"
      → compare_quarters(q1="Q1", q2="Q2")  e.g. "Q1: 6.8M NIS, Q2: 7.2M NIS"

In [ ]:

questions = [
    "What was the total revenue in March?",
    "What is the best-selling product in electronics?",
    "How did Q1 compare to Q2 in revenue?",
]






══════════════════════════════════════════════════════════════
 CONCEPT 5 — CONVERSATION MEMORY
══════════════════════════════════════════════════════════════

LLM API calls are stateless. The model forgets everything between calls.
To create a real conversation, you must send the full message history every time.

HOW IT WORKS:
  You maintain a list of messages and append to it after each turn.

  history = [
      {"role": "user",      "content": "My dataset has 80,000 rows."},
      {"role": "assistant", "content": "That's a medium-sized dataset..."},
      {"role": "user",      "content": "What should I check first?"},
  ]

  Each API call sends: [system message] + history
  The LLM sees everything — so "what you mentioned" actually refers to earlier turns.

MEMORY STRATEGIES:
  Buffer memory  — keep all messages (simple, works for short conversations)
  Window memory  — keep only the last N turns (limits token cost)
  Summary memory — summarize old turns into one message (saves tokens, loses detail)

TOKEN WARNING:
  Long conversations accumulate tokens fast.
  Each turn adds ~2× the tokens of a single call.
  Always plan a maximum conversation length in production apps.

EXAMPLE ──────────────────────────────────────────────────────

In [ ]:

print()
print("=" * 55)
print("CONCEPT 5: Conversation Memory")
print("=" * 55)

# from openai import OpenAI
# client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
#
# def chat_with_memory(user_message, history):
#     """Send a message and update conversation history."""
#     history.append({"role": "user", "content": user_message})
#     response = client.chat.completions.create(
#         model="gpt-4o-mini",
#         messages=[
#             {"role": "system", "content": "You are a data analyst assistant. Be concise."}
#         ] + history,
#         max_tokens=150
#     )
#     reply = response.choices[0].message.content
#     history.append({"role": "assistant", "content": reply})
#     return reply, history
#
# history = []
# reply, history = chat_with_memory(
#     "My dataset has hotel bookings for 2023, about 80,000 rows.", history)
# print(f"Turn 1: {reply}")
#
# reply, history = chat_with_memory(
#     "Which columns should I check first for quality issues?", history)
# print(f"Turn 2: {reply}")
#
# reply, history = chat_with_memory(
#     "You mentioned cancellations — how do I analyze that pattern?", history)
# print(f"Turn 3: {reply}")
#
# print(f"\nTotal messages in history: {len(history)}")
# print("Turn 1 = 2 messages, Turn 2 = 4 messages, Turn 3 = 6 messages")

print("Uncomment the memory code above once you have an API key.")
print()
print("Each turn appends 2 messages (user + assistant).")
print("  Turn 1 → 2 messages in history")
print("  Turn 5 → 10 messages in history")
print("  Token cost grows with every exchange — plan a max conversation length.")


══════════════════════════════════════════════════════════════
 EXERCISE 5
══════════════════════════════════════════════════════════════
Build a multi-turn conversation with a data pipeline advisor.
Use pipeline_context below as the first message.

  Turn 1: introduce the pipeline using pipeline_context
  Turn 2: ask which step in the pipeline is most likely to fail
  Turn 3: ask how to handle that failure in production
  Turn 4: ask for a code sketch of the error handling

Use chat_with_memory() from the example above.
After the conversation ends, print how many messages are in history.

Expected output:
    Turn 1: (advice about your ETL pipeline)
    Turn 2: (identifies the most fragile step)
    Turn 3: (error handling strategy for that step)
    Turn 4: (code sketch with try/except or retry logic)
    Total messages in history: 8

In [ ]:

history = []
pipeline_context = "I'm building an ETL pipeline: hourly CSV files from an FTP server, pandas cleaning, then loading into SQLite."






══════════════════════════════════════════════════════════════
 CONCEPT 6 — OBSERVABILITY: LOGGING AND TRACING LLM CALLS
══════════════════════════════════════════════════════════════

# In production, you cannot debug what you cannot see.
# Observability means recording what goes into the LLM and what comes out.
#
# What to log per call:
#   - timestamp
#   - prompt (system + user message)
#   - response
#   - model used
#   - latency (how long the call took)
#   - token count (if available)
#
# LangSmith is LangChain's built-in tracing and evaluation tool.
# To enable it, set these environment variables before running your code:
#
#   export LANGCHAIN_TRACING_V2=true
#   export LANGCHAIN_API_KEY=your_key_here
#   export LANGCHAIN_PROJECT=my-project
#
# After that, every chain, agent, and tool call is logged automatically.
# You can view the full traces at: smith.langchain.com
#
# You can also build a simple logger yourself — useful when:
#   - You do not want to share data with LangSmith
#   - You want to store traces in your own database
#   - You are building a custom monitoring dashboard

EXAMPLE ──────────────────────────────────────────────────────


In [ ]:
import time
import datetime

class TraceLogger:
    """
    Records every LLM call: prompt, response, model, latency.
    Acts as a local alternative to LangSmith for learning.
    """

    def __init__(self):
        self.traces = []

    def log(self, prompt, response, model='gpt-4o-mini', start_time=None):
        latency_ms = round((time.time() - start_time) * 1000) if start_time else 0
        trace = {
            'timestamp':  datetime.datetime.now().strftime('%H:%M:%S'),
            'model':      model,
            'prompt':     prompt[:60] + '...' if len(prompt) > 60 else prompt,
            'response':   response[:60] + '...' if len(response) > 60 else response,
            'latency_ms': latency_ms,
        }
        self.traces.append(trace)
        return trace

    def report(self):
        print(f"\n{'=' * 55}")
        print(f"  TRACE LOG  ({len(self.traces)} calls)")
        print(f"{'=' * 55}")
        for i, t in enumerate(self.traces, 1):
            print(f"  Call {i}  [{t['timestamp']}]  model={t['model']}  latency={t['latency_ms']}ms")
            print(f"    prompt:   {t['prompt']}")
            print(f"    response: {t['response']}")
        total = len(self.traces)
        avg = sum(t['latency_ms'] for t in self.traces) / total if total else 0
        print(f"{'=' * 55}")
        print(f'  Average latency: {avg:.0f}ms')
        print(f"{'=' * 55}\n")


# --- Simulated LLM call (no API key needed) ---
def fake_llm_call(prompt, model='gpt-4o-mini'):
    """Simulates an LLM response. Replace with real openai.chat.completions.create()."""
    responses = {
        'classify':  'Category: billing',
        'summarize': 'Summary: customer needs refund for order 4421',
        'translate': 'Translation: Bonjour, comment puis-je vous aider?',
    }
    time.sleep(0.05)  # simulate network latency
    for key in responses:
        if key in prompt.lower():
            return responses[key]
    return 'Response: task completed'


# --- Using the trace logger ---
logger = TraceLogger()

calls = [
    ('Classify this ticket: customer was charged twice for their order', 'gpt-4o-mini'),
    ('Summarize: customer wants a refund for order 4421, purchased on Monday', 'gpt-4o-mini'),
    ('Translate to French: Hello, how can I help you today?', 'claude-haiku'),
]

for prompt, model in calls:
    t0 = time.time()
    response = fake_llm_call(prompt, model)
    logger.log(prompt, response, model, start_time=t0)

logger.report()

══════════════════════════════════════════════════════════════
 EXERCISE 6
══════════════════════════════════════════════════════════════
# Build a trace logger that records 4 LLM support-ticket calls.
#
# Use the TraceLogger class and fake_llm_call function from above.
#
# After logging all 4 calls, print the full trace report.
#
# Expected output (latency will vary):
#   ══════...
#   TRACE LOG  (4 calls)
#   ══════...
#   Call 1  [HH:MM:SS]  model=gpt-4o-mini  latency=...ms
#     prompt:   Classify: 'My account was locked...
#     response: Response: task completed
#   ...
#   Average latency: ~50ms
#
# --- starting data ---
prompts_to_log = [
    ("Classify: 'My account was locked after too many login attempts'", 'gpt-4o-mini'),
    ('Summarize: user cannot access dashboard after password reset',     'gpt-4o-mini'),
    ('Translate to Spanish: Your issue has been resolved.',              'claude-haiku'),
    ("Classify: 'I never received my order confirmation email'",        'gpt-4o-mini'),
]

══════════════════════════════════════════════════════════════
 CONCEPT 7 — SMART SCHEMA AGENT: STAGE 5 — LLM-POWERED NL→SQL
══════════════════════════════════════════════════════════════
 CUMULATIVE PROJECT — Stage 5 builds on Stages 1-4 (W6-W9).
 The agent now uses LangChain tool calling to convert natural language
 questions into SQL queries. The LLM gets two tools: get_schema (to read
 the database structure) and run_sql (to execute the query it writes).

 Why this matters: this is the core pattern of every enterprise Text-to-SQL
 system. The LLM does not touch the database directly — it uses tools, and
 the tools enforce the safety rules from Stage 3.

 Flow: user question → LLM calls get_schema → LLM calls run_sql → answer

 The safety guarantee: run_sql only accepts SELECT queries.
 The LLM generates SQL, but it can only read — never destroy.

 EXAMPLE ──────────────────────────────────────────────────────

In [ ]:
# Install: pip install langchain langchain-openai
# Requires: OPENAI_API_KEY environment variable

import sqlite3
import json
import os
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

# Setup a demo database
_conn = sqlite3.connect(":memory:")
_cur  = _conn.cursor()
_cur.execute("CREATE TABLE employees (id INTEGER PRIMARY KEY, name TEXT, dept TEXT, salary REAL)")
_cur.executemany("INSERT INTO employees VALUES (?,?,?,?)", [
    (1, "Alice", "Engineering", 95000),
    (2, "Bob",   "Sales",       72000),
    (3, "Carol", "Finance",     81000),
    (4, "Dave",  "Engineering", 88000),
])
_conn.commit()


@tool
def get_schema() -> str:
    """Return the database schema as JSON so the LLM knows what tables exist."""
    _cur.execute("SELECT name FROM sqlite_master WHERE type='table' AND name != 'audit_log'")
    tables = [r[0] for r in _cur.fetchall()]
    schema = {}
    for t in tables:
        _cur.execute(f"PRAGMA table_info({t})")
        schema[t] = [{"col": r[1], "type": r[2]} for r in _cur.fetchall()]
    return json.dumps(schema)


@tool
def run_sql(sql: str) -> str:
    """Run a SELECT query and return results as JSON. Only SELECT is allowed."""
    if not sql.strip().upper().startswith("SELECT"):
        return json.dumps({"error": "Only SELECT queries are permitted."})
    _cur.execute(sql)
    col_names = [d[0] for d in _cur.description]
    rows = [dict(zip(col_names, row)) for row in _cur.fetchall()]
    return json.dumps(rows)


# Build the agent executor
if os.environ.get("OPENAI_API_KEY"):
    llm   = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    tools = [get_schema, run_sql]

    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a SQL expert. Call get_schema first to learn the database, then run_sql to answer the question. Write safe, read-only SELECT queries only."),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ])

    executor = AgentExecutor(
        agent=create_tool_calling_agent(llm, tools, prompt),
        tools=tools,
        verbose=False,
    )

    result = executor.invoke({"input": "Which department has the highest average salary?"})
    print(result["output"])
else:
    print("Set OPENAI_API_KEY to run this example.")
    print("Pattern: LLM calls get_schema() -> writes SELECT -> calls run_sql() -> returns answer")

_conn.close()

══════════════════════════════════════════════════════════════
 EXERCISE 7 — SMART SCHEMA AGENT: STAGE 5
══════════════════════════════════════════════════════════════
 Build the NL→SQL agent using the starting data below.
 Define get_schema and run_sql as @tool functions pointing at the sales database.
 Build the AgentExecutor with gpt-4o-mini.

 Ask the agent all 3 questions in the questions list.
 For each question, print the question and the agent's answer.

 Expected output (exact wording from the LLM will vary):
     Q: What is the total revenue by product?
     A: Laptop: $9,000.00, Monitor: $700.00, Keyboard: $89.00, Tablet: $2,100.00

     Q: Which region had the most orders?
     A: North had the most orders with 3 orders.

     Q: What is the average order amount?
     A: The average order amount is approximately $2,207.80.

 If OPENAI_API_KEY is not set, print "Set OPENAI_API_KEY to run" for each question.

 --- starting data ---

In [ ]:
import sqlite3
import json
import os
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

conn = sqlite3.connect(":memory:")
cur  = conn.cursor()
cur.execute("CREATE TABLE sales (sale_id INTEGER PRIMARY KEY, product TEXT, amount REAL, region TEXT)")
cur.executemany("INSERT INTO sales VALUES (?,?,?,?)", [
    (1, "Laptop",   4500.0, "North"),
    (2, "Monitor",   350.0, "South"),
    (3, "Keyboard",   89.0, "North"),
    (4, "Laptop",   4500.0, "East"),
    (5, "Monitor",   350.0, "North"),
    (6, "Tablet",   2100.0, "South"),
])
conn.commit()

questions = [
    "What is the total revenue by product?",
    "Which region had the most orders?",
    "What is the average order amount?",
]

# ══════════════════════════════════════════════════════════════
#  CONCEPT 8 — MCP: TYPED TOOL MANIFESTS
# ══════════════════════════════════════════════════════════════
#
#  Concept 4 showed the LangChain @tool decorator — it reads Python
#  docstrings and type hints to build a JSON schema for the LLM.
#  That schema is LangChain-specific.
#
#  MCP (Model Context Protocol) is Anthropic's open standard for
#  declaring tools in a model-agnostic, framework-agnostic JSON format.
#  The same manifest works with Claude Desktop, Cursor, VS Code agents,
#  and any custom Python agent — no LangChain required.
#
#  MCP TOOL MANIFEST STRUCTURE:
#    {
#      "name":        "get_schema",
#      "description": "Return the database schema. Call this before run_sql.",
#      "inputSchema": {
#        "type":       "object",
#        "properties": {
#          "table_name": {
#            "type":        "string",
#            "description": "Name of the table to inspect."
#          }
#        },
#        "required": ["table_name"]
#      }
#    }
#
#  WHY THIS MATTERS IN PRODUCTION:
#    - Your tool contract is a versioned JSON file, not buried in a docstring
#    - Any AI client that supports MCP can call your tool without modification
#    - Teams share tool registries: data team writes tools, AI team consumes them
#    - Safety constraints live in the manifest (required fields, allowed types)
#
#  IN LANGCHAIN: use StructuredTool(name, description, args_schema, func)
#    This registers a tool from an explicit schema instead of a decorator.
#
#  INSTALL: pip install langchain langchain-core pydantic
#
#  EXAMPLE ----------------------------------------------------------

In [ ]:
print()
print("=" * 55)
print("CONCEPT 8: MCP Typed Tool Manifest")
print("=" * 55)

import json
from pydantic import BaseModel
from langchain.tools import StructuredTool

# The @tool decorator (Concept 4) builds a JSON schema from docstrings and type hints.
# MCP (Model Context Protocol) makes that schema explicit and model-agnostic.
# The same manifest works with Claude Desktop, Cursor, and any custom agent.

# --- What @tool generates internally for run_sql ---
schema_from_tool_decorator = {
    "name":        "run_sql",
    "description": "Run a SELECT query and return results as JSON. Only SELECT is allowed.",
    "parameters":  {
        "type":       "object",
        "properties": {"sql": {"type": "string"}},
        "required":   ["sql"],
    },
}

# --- The same contract written as an explicit MCP manifest ---
mcp_manifest = {
    "name":        "run_sql",
    "description": "Execute a read-only SQL SELECT query on the sales database.",
    "inputSchema": {
        "type":       "object",
        "properties": {
            "sql": {
                "type":        "string",
                "description": "A valid SQL SELECT query. Only SELECT is permitted.",
            }
        },
        "required": ["sql"],
    },
}

print("MCP manifest:")
print(json.dumps(mcp_manifest, indent=2))
print()

# --- Register an MCP-compatible tool in LangChain using StructuredTool ---
# No @tool decorator needed — the manifest drives the registration.

class RunSQLInput(BaseModel):
    sql: str

def _run_sql(sql: str) -> str:
    if not sql.strip().upper().startswith("SELECT"):
        return '{"error": "Only SELECT is permitted"}'
    return '{"rows": [{"dept": "Engineering", "avg_salary": 91500}]}'

sql_tool = StructuredTool(
    name        = mcp_manifest["name"],
    description = mcp_manifest["description"],
    args_schema = RunSQLInput,
    func        = _run_sql,
)

result = sql_tool.run({"sql": "SELECT dept, AVG(salary) FROM employees GROUP BY dept"})
print(f"Tool name:   {sql_tool.name}")
print(f"Tool result: {result}")
print()
print("Schema the LLM sees (args_schema.model_fields):")
for field, info in RunSQLInput.model_fields.items():
    print(f"  {field}: {info.annotation.__name__}")
print()
print("Why MCP matters:")
print("  @tool  — LangChain-specific, reads Python docstrings")
print("  MCP    — framework-agnostic JSON manifest, same tool works in")
print("           Claude Desktop, Cursor, custom agents, across teams")

# ══════════════════════════════════════════════════════════════
#  EXERCISE 8 — MCP TYPED TOOL MANIFEST
# ══════════════════════════════════════════════════════════════
#
#  You have two tools that monitor data pipelines.
#  Define an MCP manifest for each and register them with LangChain.
#
#  1. Fill in MANIFEST_STATUS and MANIFEST_ROWCOUNT:
#       description — plain English: what the tool does, when to call it
#       inputSchema — {"type": "object", "properties": {...}, "required": [...]}
#
#  2. Define Python functions:
#       get_pipeline_status(job_id: str) -> str
#         Look up job_id in PIPELINE_STATUS_DB. Return the status string.
#         If not found, return "UNKNOWN job_id"
#       get_row_count(table_name: str) -> str
#         Look up table_name in TABLE_ROW_COUNTS. Return "{table}: {n} rows"
#         If not found, return "Table not found"
#
#  3. Register both using StructuredTool — use manifest name and description.
#     Define a Pydantic input model for each (1 field each).
#
#  4. Call each tool and print the result.
#     Then print the args_schema.schema() for one tool so you can see
#     the JSON Schema that the LLM would receive.
#
#  Expected output:
#      Manifest: get_pipeline_status
#        input: job_id (string)
#      Manifest: get_row_count
#        input: table_name (string)
#
#      get_pipeline_status("job-2024-001"): SUCCESS — 15,420 rows processed
#      get_row_count("employees"):          employees: 42 rows
#
#      Schema exposed to LLM:
#        {'job_id': {'title': 'Job Id', 'type': 'string'}}
#
# --- starting data ---

In [ ]:
from pydantic import BaseModel
from langchain.tools import StructuredTool

# Hardcoded data stores (no live database needed)
PIPELINE_STATUS_DB = {
    "job-2024-001": "SUCCESS — 15,420 rows processed",
    "job-2024-002": "FAILED  — connection timeout at step 3",
    "job-2024-003": "RUNNING — step 2 of 4",
}

TABLE_ROW_COUNTS = {
    "employees": 42,
    "orders":    8750,
    "products":  193,
}

# MCP manifest skeleton — fill in description and inputSchema:
MANIFEST_STATUS = {
    "name":        "get_pipeline_status",
    "description": "",    # describe what the tool does and when to call it
    "inputSchema": {},    # JSON Schema: type, properties, required
}

MANIFEST_ROWCOUNT = {
    "name":        "get_row_count",
    "description": "",
    "inputSchema": {},
}